In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from utils import aproximar_limite_valido 

In [2]:
df = pd.read_csv("../data/comprovantes_pix_10000_anomalias.csv", sep=";")

In [3]:
df.agg(['min', 'max'])

,EndToEndId,DataHora,Valor,Moeda,Pagador_Nome,Pagador_CPF_CNPJ,Pagador_Banco,Recebedor_Nome,Recebedor_CPF_CNPJ,Recebedor_Banco,ChavePix_Utilizada,TipoChave,Descricao,Status,Anomalia
min,00032a00-5360-40da-aa43-ea59ce48593e,2025-02-30 25:61:00,0.00,BRL,Agatha Almeida,000.000.000-00,BTG Pactual,Agatha Araújo,10.127.878/0001-59,BTG Pactual,+551091035-2412,CNPJ,Pagamento referente ao serviço 1,Concluída,0
max,ffffad37-8e79-4cd8-a1b6-b05d20bc5b5e,2025-08-20 15:44:56,4999.79,BRL,Yuri da Rosa,999.790.314-14,Santander Brasil,Yuri das Neves,999.704.956-96,Santander Brasil,user9@email.com,Telefone,Pagamento referente ao serviço 999,Pendente,1


In [4]:
df.isnull().sum()

EndToEndId            0
DataHora              0
Valor                 0
Moeda                 0
Pagador_Nome          0
Pagador_CPF_CNPJ      0
Pagador_Banco         0
Recebedor_Nome        0
Recebedor_CPF_CNPJ    0
Recebedor_Banco       0
ChavePix_Utilizada    0
TipoChave             0
Descricao             0
Status                0
Anomalia              0
dtype: int64

In [5]:
# Função de tratamento
df['DataHora_Tratada'] = df['DataHora'].apply(aproximar_limite_valido)

# Extração de atributos temporais
df['Hora'] = df['DataHora_Tratada'].dt.hour
df['DiaDaSemana'] = df['DataHora_Tratada'].dt.dayofweek
df['FimDeSemana'] = df['DiaDaSemana'].apply(lambda x: 1 if x >= 5 else 0)
df['Dia_do_Mes'] = df['DataHora_Tratada'].dt.day

In [6]:
# Criando flags comportamentais
df['Horario_Comercial'] = df.apply(lambda row: 1 if (8 <= row['Hora'] <= 18) and (row['FimDeSemana'] == 0) else 0, axis=1)
df['Madrugada'] = df['Hora'].apply(lambda x: 1 if 0 <= x <= 5 else 0)
df['Dia_de_Pagamento'] = df['Dia_do_Mes'].apply(lambda x: 1 if x in [5, 6, 7, 20, 21, 22] else 0)
df['Valor_Redondo'] = df['Valor'].apply(lambda x: 1 if x % 1 == 0 else 0)
df['Status_Pendente'] = (df['Status'] == 'Pendente').astype(int)
df['Mesmo_Banco'] = (df['Pagador_Banco'] == df['Recebedor_Banco']).astype(int)

In [7]:
# Remove colunas que não serão usadas na modelagem
cols_to_drop = ['Descricao', 'DataHora', 'Moeda', 'Pagador_Nome', 'Pagador_CPF_CNPJ', 
                'Pagador_Banco', 'Recebedor_Banco', 'Recebedor_Nome', 'Recebedor_CPF_CNPJ', 
                'ChavePix_Utilizada', 'DataHora_Tratada', 'EndToEndId']
df_limpo = df.drop(columns=cols_to_drop, errors='ignore')

In [ ]:
# Verificação de qualidade
print("Resumo do pré-processamento:")
display(df_limpo.head())
print(f"Total de registros: {df_limpo.shape[0]} | Colunas: {df_limpo.shape[1]}")

# Identificando as colunas categóricas no df_limpo
# As colunas que não são numéricas ou flags binárias precisam ser tratadas
cols_categoricas = ['TipoChave', 'Status']

# Criando a lista de índices para o SMOTE
indices_categoricos = [df_limpo.columns.get_loc(col) for col in cols_categoricas]

print(f"Índices das colunas categóricas para o SMOTE: {indices_categoricos}")

# Salvar a base pronta para a modelagem
df_limpo.to_csv("../data/base_tratada_final.csv", index=False)

Resumo do pré-processamento:


,Valor,TipoChave,Status,Anomalia,Hora,DiaDaSemana,FimDeSemana,Dia_do_Mes,Horario_Comercial,Madrugada,Dia_de_Pagamento,Valor_Redondo,Status_Pendente,Mesmo_Banco
0,4658.86,Telefone,Concluída,0,21,3,0,26,0,0,0,0,0,0
1,3184.72,Telefone,Pendente,0,17,0,0,23,1,0,0,0,1,0
2,1054.48,Chave Aleatória,Pendente,0,15,4,0,27,1,0,0,0,1,0
3,3564.76,E-mail,Estornada,0,13,1,0,15,1,0,0,0,0,0
4,15.16,E-mail,Pendente,0,6,0,0,23,0,0,0,0,1,0


Total de registros: 10000 | Colunas: 14
Índices das colunas categóricas para o SMOTE: [1, 2]


In [9]:
df['Anomalia'].value_counts()

Anomalia
0    9900
1     100
Name: count, dtype: int64